# Part I — Sector Coupling: Electricity + Heat (DNK-SWE-NOR-DEU)

Extends the Part D multi-country model by coupling the electricity sector with the **heating sector**.  
Each country gets a heat bus, a heat demand load, **heat pumps** (electricity→heat,
temperature-dependent COP), and **gas boilers** as backup heating.

1. **Section 1 (Optimisation)** — run once, saves results to `plots_part_i/`
2. **Section 2 (Plotting)** — loads from CSV, re-run freely without the solver

---
## Section 1 — Optimisation
*Run these cells once. Skip after CSVs exist.*

In [ ]:
import os
import pandas as pd
import pypsa

df_elec = pd.read_csv('data/electricity_demand.csv', sep=';', index_col=0)
df_elec.index = pd.to_datetime(df_elec.index)
df_onshorewind = pd.read_csv('data/onshore_wind_1979-2017.csv', sep=';', index_col=0)
df_onshorewind.index = pd.to_datetime(df_onshorewind.index)
df_offshorewind = pd.read_csv('data/offshore_wind_1979-2017.csv', sep=';', index_col=0)
df_offshorewind.index = pd.to_datetime(df_offshorewind.index)
df_solar = pd.read_csv('data/pv_optimal.csv', sep=';', index_col=0)
df_solar.index = pd.to_datetime(df_solar.index)
df_heat = pd.read_csv('data/heat_demand.csv', sep=';', index_col=0)
df_heat.index = pd.to_datetime(df_heat.index)

# Temperature: mixed format (hourly + some daily rows), parse as UTC then strip to naive
df_temp = pd.read_csv('data/temperature_20260429.csv', sep=';', index_col=0)
if 'time' in df_temp.index:
    df_temp = df_temp.drop('time')
df_temp.index = pd.to_datetime(df_temp.index, format='mixed', utc=True)
df_temp.index = df_temp.index.tz_localize(None)  # make naive to match snapshot alignment
df_temp = df_temp.loc[~df_temp.index.duplicated(keep='first')].sort_index()
for col in df_temp.columns:
    df_temp[col] = pd.to_numeric(df_temp[col], errors='coerce')
df_temp = df_temp.ffill().bfill()
df_temp = df_temp.resample('h').ffill()  # expand daily rows to hourly

countries = ["DNK", "SWE", "NOR", "DEU"]
gas_price  = 30.0

def annuity(n, r):
    return r / (1. - 1. / (1. + r) ** n) if r > 0 else 1 / n

In [ ]:
network = pypsa.Network()
hours_in_2015 = pd.date_range('2015-01-01 00:00Z', '2015-12-31 23:00Z', freq='h')
network.set_snapshots(hours_in_2015.values)

# Naive hourly snapshots for temperature alignment
# (network.snapshots stores .values which strips UTC timezone from the DatetimeIndex)
naive_snapshots = pd.date_range('2015-01-01', periods=8760, freq='h')

# Buses, electricity loads, heat buses, heat loads
# Note: use .values directly — df_elec/df_heat cover exactly 2015 in order.
# .reindex(network.snapshots) would return all-NaN due to UTC vs naive mismatch.
for c in countries:
    network.add("Bus", c, v_nom=400)
    network.add("Bus", f"{c} heat", carrier="heat")
    network.add("Load", f"electricity_demand_{c}", bus=c,
                p_set=df_elec[c].values)
    network.add("Load", f"heat_demand_{c}", bus=f"{c} heat",
                p_set=df_heat[c].values)

for carrier, co2 in [
    ("gas", 0.19), ("onshorewind", 0), ("offshorewind", 0),
    ("solar", 0), ("battery storage", 0), ("pumped hydro", 0),
    ("heat", 0), ("heat pump", 0),
]:
    if carrier not in network.carriers.index:
        network.add("Carrier", carrier, co2_emissions=co2)

In [ ]:
def add_country_generators(network, country):
    CF_on  = df_onshorewind[country][[h.strftime("%Y-%m-%dT%H:%M:%SZ") for h in network.snapshots]]
    CF_off = df_offshorewind[country][[h.strftime("%Y-%m-%dT%H:%M:%SZ") for h in network.snapshots]]
    CF_pv  = df_solar[country][[h.strftime("%Y-%m-%dT%H:%M:%SZ") for h in network.snapshots]]

    network.add("Generator", f"Onshore wind {country}", bus=country, carrier="onshorewind",
                p_nom_extendable=True, capital_cost=annuity(27, 0.07) * 1118775,
                marginal_cost=0, p_max_pu=CF_on.values)
    network.add("Generator", f"Offshore wind {country}", bus=country, carrier="offshorewind",
                p_nom_extendable=True, capital_cost=annuity(27, 0.07) * 2115944,
                marginal_cost=0, p_max_pu=CF_off.values)
    network.add("Generator", f"Solar {country}", bus=country, carrier="solar",
                p_nom_extendable=True, capital_cost=annuity(25, 0.07) * 450000,
                marginal_cost=0, p_max_pu=CF_pv.values)
    network.add("Generator", f"OCGT {country}", bus=country, carrier="gas",
                p_nom_extendable=True, capital_cost=annuity(25, 0.07) * 453960,
                marginal_cost=gas_price / 0.41)
    network.add("Generator", f"CCGT {country}", bus=country, carrier="gas",
                p_nom_extendable=True, capital_cost=annuity(25, 0.07) * 880000,
                marginal_cost=gas_price / 0.56)
    network.add("StorageUnit", f"battery storage {country}", bus=country,
                carrier="battery storage", p_nom_extendable=True, max_hours=2,
                capital_cost=annuity(20, 0.07) * 2 * 288000,
                efficiency_store=0.98, efficiency_dispatch=0.97,
                cyclic_state_of_charge=True)

for c in countries:
    add_country_generators(network, c)

In [ ]:
def load_hydro_inflow(csv_path, year=2012):
    df = pd.read_csv(csv_path)
    df = df[df["Year"] == year].copy()
    df['date'] = pd.to_datetime(df[['Year', 'Month', 'Day']])
    return df.set_index('date')['Inflow [GWh]'].reindex(naive_snapshots).fillna(0).values

for country, csv, p_nom_max in [
    ("SWE", 'data/inflow/Hydro_Inflow_SE.csv', 16000),
    ("NOR", 'data/inflow/Hydro_Inflow_NO.csv', 33000),
    ("DEU", 'data/inflow/Hydro_Inflow_DE.csv',  7000),
]:
    network.add("StorageUnit", f"pumped hydro {country}",
                bus=country, carrier="pumped hydro",
                p_nom_extendable=True, p_nom_max=p_nom_max, max_hours=8,
                capital_cost=annuity(80, 0.07) * 400000,
                efficiency_store=0.9, efficiency_dispatch=0.9,
                cyclic_state_of_charge=True, marginal_cost=1,
                inflow=load_hydro_inflow(csv))

In [ ]:
country_temp_col = {"DNK": "DK", "SWE": "SE", "NOR": "NO", "DEU": "DE"}

def cop(t_source, t_sink=55):
    delta_t = t_sink - t_source
    return 6.81 - 0.121 * delta_t + 0.00063 * delta_t ** 2

for c in countries:
    temp_series = df_temp[country_temp_col[c]].reindex(naive_snapshots, method='ffill').ffill().bfill()
    cop_values = cop(temp_series).clip(lower=1.0).values

    # Heat pump: electricity bus → heat bus, efficiency = COP (varies hourly with temperature)
    # Capital cost: ~986 000 €/MW_e, 33-year lifetime
    network.add("Link", f"heat pump {c}",
                bus0=c, bus1=f"{c} heat", carrier="heat pump",
                p_nom_extendable=True,
                capital_cost=annuity(33, 0.07) * 986361,
                efficiency=cop_values)

    # Gas boiler: backup heat, DEA large gas boiler (62 000 €/MW_th, 20 yr, η=0.90)
    network.add("Generator", f"gas boiler {c}",
                bus=f"{c} heat", carrier="gas",
                p_nom_extendable=True,
                capital_cost=annuity(20, 0.07) * 62000,
                marginal_cost=gas_price / 0.9)

In [ ]:
x_line = 0.1
for name, b0, b1, s_nom in [
    ("DK-NO", "DNK", "NOR", 1632),
    ("DK-SE", "DNK", "SWE", 2415),
    ("DK-DE", "DNK", "DEU", 3500),
    ("SE-NO", "SWE", "NOR", 3945),
    ("SE-DE", "SWE", "DEU",  615),
    ("NO-DE", "NOR", "DEU", 1400),
]:
    network.add("Line", name, bus0=b0, bus1=b1, x=x_line, s_nom=s_nom)

In [ ]:
network.optimize(solver_name="gurobi", solver_options={"OutputFlag": 0, "LogToConsole": 0})
print(f"Total system cost: {network.objective / 1e6:.2f} M\u20ac/yr")

In [ ]:
import re
os.makedirs('plots_part_i', exist_ok=True)

ELEC_TECHS = ["Onshore wind", "Offshore wind", "Solar", "OCGT", "CCGT",
              "battery storage", "pumped hydro"]

# Electricity capacity table [GW]
elec_gens = [c for c in network.generators.index if not c.startswith("gas boiler")]
elec_caps = pd.concat([
    network.generators.p_nom_opt[elec_gens],
    network.storage_units.p_nom_opt,
]) / 1000
cap_elec = (
    elec_caps.rename_axis("name").reset_index(name="capacity")
    .assign(
        country=lambda df: df["name"].str.split().str[-1],
        tech=lambda df: df["name"].str.replace(r"\s+(DNK|SWE|NOR|DEU)$", "", regex=True)
    )
    .pivot(index="country", columns="tech", values="capacity")
    .reindex(index=["DNK", "SWE", "NOR", "DEU"], columns=ELEC_TECHS)
    .fillna(0)
    .rename(index={"DNK": "Denmark", "SWE": "Sweden", "NOR": "Norway", "DEU": "Germany"})
)
cap_elec.to_csv('plots_part_i/cap_elec_table.csv')

# Heat capacity table [GW]
hp_caps = network.links.p_nom_opt[
    [c for c in network.links.index if c.startswith("heat pump ")]].copy() / 1000
hp_caps.index = hp_caps.index.str.replace(r"heat pump\s+(DNK|SWE|NOR|DEU)$", r"\1", regex=True)
boiler_caps = network.generators.p_nom_opt[
    [c for c in network.generators.index if c.startswith("gas boiler ")]].copy() / 1000
boiler_caps.index = boiler_caps.index.str.replace(
    r"gas boiler\s+(DNK|SWE|NOR|DEU)$", r"\1", regex=True)
cap_heat = pd.DataFrame({"heat pump": hp_caps, "gas boiler": boiler_caps}).reindex(
    ["DNK", "SWE", "NOR", "DEU"]).fillna(0)
cap_heat.rename(index={"DNK": "Denmark", "SWE": "Sweden",
                        "NOR": "Norway", "DEU": "Germany"}, inplace=True)
cap_heat.to_csv('plots_part_i/cap_heat_table.csv')

# Annual electricity generation mix [TWh]
gen_by_name = network.generators_t.p[elec_gens].sum() / 1e6
gen_by_name.index = gen_by_name.index.str.replace(r"\s+(DNK|SWE|NOR|DEU)$", "", regex=True)
gen_by_tech = gen_by_name.groupby(level=0).sum()
sto_by_name = network.storage_units_t.p.clip(lower=0).sum() / 1e6
sto_by_name.index = sto_by_name.index.str.replace(r"\s+(DNK|SWE|NOR|DEU)$", "", regex=True)
sto_by_tech = sto_by_name.groupby(level=0).sum()
generation_mix = pd.concat([gen_by_tech, sto_by_tech]).reindex(ELEC_TECHS).fillna(0)
generation_mix.to_csv('plots_part_i/generation_mix.csv')

# Annual heat supply mix [TWh] per country
hp_cols = [c for c in network.links.index if c.startswith("heat pump ")]
hp_heat = network.links_t.p1[hp_cols].sum() / 1e6
hp_heat.index = hp_heat.index.str.replace(r"heat pump\s+(DNK|SWE|NOR|DEU)$", r"\1", regex=True)
boiler_cols = [c for c in network.generators.index if c.startswith("gas boiler ")]
boiler_heat = network.generators_t.p[boiler_cols].sum() / 1e6
boiler_heat.index = boiler_heat.index.str.replace(
    r"gas boiler\s+(DNK|SWE|NOR|DEU)$", r"\1", regex=True)
heat_supply = pd.DataFrame({"heat pump": hp_heat, "gas boiler": boiler_heat}).reindex(
    ["DNK", "SWE", "NOR", "DEU"]).fillna(0)
heat_supply.rename(index={"DNK": "Denmark", "SWE": "Sweden",
                           "NOR": "Norway", "DEU": "Germany"}, inplace=True)
heat_supply.to_csv('plots_part_i/heat_supply_mix.csv')

# Hourly electricity dispatch by tech [MW]
gen_disp = network.generators_t.p[elec_gens].copy()
gen_disp.columns = gen_disp.columns.str.replace(r"\s+(DNK|SWE|NOR|DEU)$", "", regex=True)
gen_t = gen_disp.T.groupby(level=0).sum().T
sto_disp = network.storage_units_t.p.copy()
sto_disp.columns = sto_disp.columns.str.replace(
    r"(battery storage|pumped hydro)\s+(DNK|SWE|NOR|DEU)$", r"\1", regex=True)
sto_t = sto_disp.T.groupby(level=0).sum().T
dispatch_by_tech = pd.concat([gen_t, sto_t], axis=1).T.groupby(level=0).sum().T
dispatch_by_tech.to_csv('plots_part_i/dispatch_by_tech.csv')

# DNK heat dispatch [MW]
hp_dnk = network.links_t.p1.get("heat pump DNK", pd.Series(0, index=network.snapshots))
boiler_dnk = network.generators_t.p.get("gas boiler DNK", pd.Series(0, index=network.snapshots))
pd.DataFrame({"heat pump": hp_dnk, "gas boiler": boiler_dnk}).to_csv(
    'plots_part_i/heat_dispatch_dnk.csv')

pd.Series({"system_cost_M_eur": network.objective / 1e6}).to_csv('plots_part_i/co2_results.csv')

print("Saved CSVs to plots_part_i/")
print("\nElectricity capacity [GW]:")
print(cap_elec)
print("\nHeat capacity [GW]:")
print(cap_heat)
print("\nGeneration mix [TWh]:")
print(generation_mix)
print("\nHeat supply [TWh]:")
print(heat_supply)

---
## Section 2 — Plotting
*Start here after CSVs exist. No solver needed.*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

cap_elec       = pd.read_csv('plots_part_i/cap_elec_table.csv', index_col=0)
cap_heat       = pd.read_csv('plots_part_i/cap_heat_table.csv', index_col=0)
generation_mix = pd.read_csv('plots_part_i/generation_mix.csv', index_col=0, header=None).squeeze()
heat_supply    = pd.read_csv('plots_part_i/heat_supply_mix.csv', index_col=0)
dispatch       = pd.read_csv('plots_part_i/dispatch_by_tech.csv', index_col=0, parse_dates=True)
heat_dnk       = pd.read_csv('plots_part_i/heat_dispatch_dnk.csv', index_col=0, parse_dates=True)
co2_res        = pd.read_csv('plots_part_i/co2_results.csv', index_col=0, header=None).squeeze()

ELEC_TECHS   = ["Onshore wind", "Offshore wind", "Solar", "OCGT", "CCGT",
                "battery storage", "pumped hydro"]
ELEC_LABELS  = ['Onshore wind', 'Offshore wind', 'Solar PV', 'Gas (OCGT)', 'Gas (CCGT)',
                'Battery storage', 'Hydro']
ELEC_COLORS  = ['blue', 'dodgerblue', 'orange', 'crimson', 'darkviolet', 'lightgreen', 'pink']
HEAT_LABELS  = ['Heat pumps', 'Gas boilers']
HEAT_COLORS  = ['tomato', 'saddlebrown']

print(f"System cost: {co2_res['system_cost_M_eur']:.1f} M\u20ac/yr")
print("\nElectricity capacity [GW]:")
print(cap_elec)
print("\nHeat capacity [GW]:")
print(cap_heat)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6), dpi=150)
cap_elec[ELEC_TECHS].plot(kind="bar", stacked=True, colors=ELEC_COLORS, rot=0, ax=ax)
ax.set_ylabel("Installed capacity [GW]")
ax.set_xlabel("")
ax.set_title("Electricity sector — installed capacity by country", fontweight='bold')
ax.legend(ELEC_LABELS, bbox_to_anchor=(1.01, 1), frameon=False)
plt.tight_layout()
plt.savefig('plots_part_i/1i_elec_capacities.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5), dpi=150)
cap_heat[['heat pump', 'gas boiler']].plot(kind="bar", stacked=True,
                                           colors=HEAT_COLORS, rot=0, ax=ax)
ax.set_ylabel("Installed capacity [GW]")
ax.set_xlabel("")
ax.set_title("Heat sector — installed capacity by country", fontweight='bold')
ax.legend(HEAT_LABELS, bbox_to_anchor=(1.01, 1), frameon=False)
plt.tight_layout()
plt.savefig('plots_part_i/1i_heat_capacities.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
gen_vals  = generation_mix.reindex(ELEC_TECHS).fillna(0)
perc_gen  = 100 * gen_vals / gen_vals.sum()

heat_total = heat_supply.sum()
perc_heat  = 100 * heat_total / heat_total.sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), dpi=150)

ax1.pie(gen_vals, colors=ELEC_COLORS,
        labels=[f'{l}\n{p:.1f}%' for l, p in zip(ELEC_LABELS, perc_gen)],
        startangle=90, labeldistance=1.18, wedgeprops={'linewidth': 0})
ax1.set_title('Annual electricity generation mix', fontweight='bold')

ax2.pie(heat_total, colors=HEAT_COLORS,
        labels=[f'{l}\n{p:.1f}%' for l, p in zip(HEAT_LABELS, perc_heat)],
        startangle=90, labeldistance=1.18, wedgeprops={'linewidth': 0})
ax2.set_title('Annual heat supply mix (all countries)', fontweight='bold')

plt.tight_layout()
plt.savefig('plots_part_i/1i_mix_pies.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5), dpi=150)
for tech, label, color in zip(ELEC_TECHS, ELEC_LABELS, ELEC_COLORS):
    if tech in dispatch.columns:
        sorted_d = dispatch[tech].sort_values(ascending=False).reset_index(drop=True)
        plt.plot(sorted_d.values, label=label, color=color, lw=2.5)
plt.axhline(0, color='black', lw=0.8)
plt.ylabel('Dispatch [MWh/h]')
plt.xlabel('Hours')
plt.title('Duration curve of electricity generation and storage')
plt.legend(fancybox=True, shadow=True, loc='best', fontsize=8)
plt.tight_layout()
plt.savefig('plots_part_i/1i_duration_curve.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
for start, end, season, fname in [
    (0,    168,  'Winter (Jan)', '1i_heat_dispatch_winter.png'),
    (4344, 4512, 'Summer (Jul)', '1i_heat_dispatch_summer.png'),
]:
    fig, ax = plt.subplots(figsize=(10, 4), dpi=150)
    heat_dnk['heat pump'].iloc[start:end].plot(ax=ax, color='tomato', lw=2,
                                               label='Heat pump output')
    heat_dnk['gas boiler'].iloc[start:end].plot(ax=ax, color='saddlebrown', lw=2,
                                                label='Gas boiler output')
    ax.set_ylabel('MW')
    ax.set_xlabel('Hour')
    ax.set_title(f'Denmark heat supply — {season}', fontweight='bold')
    ax.legend(frameon=False)
    ax.grid(True, linestyle='--', alpha=0.4)
    plt.tight_layout()
    plt.savefig(f'plots_part_i/{fname}', dpi=300, bbox_inches='tight')
    plt.show()